# Modelo de refletividade 2D com difrações

Resumo da metodologia implementada em [`processamento/refletividade_2d_difracoes.py`](../processamento/refletividade_2d_difracoes.py), com as funções auxiliares em [`processamento/utils.py`](../processamento/utils.py).

O objetivo é gerar, para um mesmo modelo, uma seção só com reflexões, uma só com difrações e o campo de onda completo (soma das duas), seguindo as etapas:

1. gerar a refletividade das reflexões (mesma metodologia de [`modelo_refletividade_2d.ipynb`](./modelo_refletividade_2d.ipynb));
2. gerar a refletividade das difrações, com hipérboles cujos ápices ficam sobre os refletores;
3. convolver reflexões e difrações com wavelets Ricker distintas (as difrações com frequências mais altas);
4. somar as duas seções e adicionar ruído gaussiano.

## Parâmetros

| Parâmetro | Valor | Descrição |
|---|---|---|
| `ns` | 256 | amostras por traço |
| `ntr` | 256 | número de traços |
| `nrefl` | 5 | número de refletores |
| `dt` | 0,006 s | intervalo de amostragem |
| `dx` | 12,5 m | espaçamento entre traços |
| `ndif` | 5 – 20 | faixa do número de difrações |
| `vel` | 1500 – 5000 m/s | faixa de velocidades (controla a abertura das hipérboles) |

Todos os sorteios usam um único gerador `rng = np.random.default_rng()`; $\mathcal{U}(a,b)$ denota distribuição uniforme.

## 1. Refletividade das reflexões

A refletividade $r(t, x)$ é gerada por `modelo_refletividade`, importada de `refletividade_2d.py`: refletores com amplitudes $a_k \sim \mathcal{U}(-1, 1)$ posicionados entre duas tendências senoidais (topo e base). As fórmulas estão em [`modelo_refletividade_2d.ipynb`](./modelo_refletividade_2d.ipynb).

Essa refletividade é usada também para posicionar os ápices das difrações.

## 2. Refletividade das difrações

O número de hipérboles é sorteado como um inteiro

$$
n_{dif} \sim \mathcal{U}\{5, \dots, 20\}
$$

e, para cada hipérbole, são sorteados posição do ápice, abertura e amplitude.

### Posição do ápice

O ápice $(x_0, t_0)$ é posicionado **sobre um refletor**, já que geologicamente as difrações nascem em descontinuidades dos próprios refletores (bordas, terminações, falhas):

- $x_0 \sim \mathcal{U}\{0, \dots, n_{tr}-1\}$: traço do ápice;
- $t_0$ (em amostras): sorteado entre as amostras do traço $x_0$ que contêm refletor,

$$
t_0 \in \big\{\, i \ge 1 \;:\; |r(i, x_0)| > 0{,}05 \,\big\}
$$

A amostra $i = 0$ é ignorada para evitar $t_0 = 0$ (hipérbole degenerada). Se o traço não tiver nenhuma amostra candidata, a hipérbole é descartada.

### Tempo de trânsito (hipérbole)

Para um difrator pontual no domínio *zero-offset*, à profundidade $z = v\,T_0/2$, o tempo de ida e volta até o traço $x$ (distância horizontal $h = (x - x_0)\,\Delta x$) é

$$
t_d(x) = \frac{2\sqrt{z^2 + h^2}}{v} = \sqrt{T_0^2 + \left(\frac{2\,(x - x_0)\,\Delta x}{v}\right)^2}
$$

com $T_0 = t_0\,\Delta t$ o tempo do ápice em segundos. No código o resultado é dividido por $\Delta t$ para voltar a amostras.

A **abertura** da hipérbole é controlada pela velocidade $v \sim \mathcal{U}(1500,\ 5000)$ m/s: longe do ápice, $t_d$ cresce com inclinação $2/v$, então

- $v$ baixa: hipérbole fechada (flancos íngremes);
- $v$ alta: hipérbole aberta (flancos suaves).

### Amplitude e decaimento

A amplitude da difração é bem menor que a das reflexões e tem sinal aleatório:

$$
a_d = \sigma\,A, \qquad \sigma \in \{-1, 1\}, \quad A \sim \mathcal{U}(0{,}05,\ 0{,}3)
$$

A energia decresce com a distância ao ápice, combinando espalhamento geométrico e uma atenuação lateral gaussiana:

$$
D(x) = \frac{t_0}{t_d(x)}\, \exp\!\left[-\frac{(x - x_0)^2}{2L^2}\right], \qquad L \sim \mathcal{U}(0{,}05,\ 0{,}3)\,n_{tr}
$$

- $t_0 / t_d(x)$: igual a 1 no ápice e menor que 1 nos flancos;
- $L$: largura (em traços) da janela gaussiana; quanto menor, mais curta a hipérbole.

### Distribuição sub-amostral

A amplitude $a_d\,D(x)$ é dividida entre as duas amostras vizinhas de $t_d(x)$, com o mesmo esquema da refletividade das reflexões:

$$
i_0 = \lfloor t_d(x) \rfloor, \qquad w = t_d(x) - i_0
$$

$$
d(i_0, x) \mathrel{+}= a_d\,D(x)\,(1 - w), \qquad d(i_0 + 1, x) \mathrel{+}= a_d\,D(x)\,w
$$

Índices fora de $[0, n_s)$ são descartados e hipérboles que se cruzam se somam.

## 3. Convolução com wavelets Ricker

Reflexões e difrações são convolvidas com wavelets Ricker **diferentes** (mesma fórmula de [`modelo_refletividade_2d.ipynb`](./modelo_refletividade_2d.ipynb)), pois as difrações têm frequências mais altas que as reflexões:

| Seção | Frequência de pico $f_p$ | Ganho $g$ |
|---|---|---|
| reflexões | $\mathcal{U}(10,\ 40)$ Hz | $\mathcal{U}(0{,}5,\ 1{,}5)$ |
| difrações | $\mathcal{U}(40,\ 70)$ Hz | $\mathcal{U}(0{,}5,\ 1{,}5)$ |

$$
s_r(t, x) = r(t, x) * g_r\,w_r(t), \qquad s_d(t, x) = d(t, x) * g_d\,w_d(t)
$$

O ganho $g$ varia a energia de cada wavelet. A frequência máxima de 70 Hz fica abaixo da frequência de Nyquist, $f_N = 1/(2\,\Delta t) \approx 83$ Hz.

## 4. Campo de onda completo e ruído

O campo de onda completo é a soma das duas seções:

$$
s(t, x) = s_r(t, x) + s_d(t, x)
$$

Sobre ele é adicionado ruído gaussiano com $\text{SNR} \sim \mathcal{U}(5,\ 20)$ dB, com as mesmas fórmulas de [`modelo_refletividade_2d.ipynb`](./modelo_refletividade_2d.ipynb):

$$
\tilde{s}(t,x) = s(t,x) + n(t,x), \qquad n \sim \mathcal{N}\!\left(0,\ \frac{P_s}{10^{\,\text{SNR}/10}}\right)
$$

Como as amplitudes das difrações são baixas, elas ficam pouco visíveis no campo completo, principalmente na versão ruidosa.